# Exemplo de Inferência — V8

Roda o modelo V8 em uma única imagem do conjunto de teste e exibe a resposta completa.
Útil para selecionar um exemplo representativo para o artigo.

### Configuração de ambiente

In [ ]:
from os import environ

environ['CUDA_VISIBLE_DEVICES'] = input('GPU ID: ')

### Imports

In [ ]:
from os.path import join
from json import load

from PIL import Image
from IPython.display import display as ipython_display
from unsloth import FastVisionModel

from scripts.authentication import authenticate_huggingface
from scripts.messages import add_inference_message, format_prompt
from scripts.data import SimpleLesionData, SimpleDatasetAnalysis

import scripts.definitions as defs

### Autenticação

In [ ]:
authenticate_huggingface()

### Configurações

In [ ]:
MODEL = 'LLaDerm-V8-11B-4bit'
TEMPERATURE = 0.005

with open(join(defs.TRAINING_PATH, 'models.json'), 'r', encoding='utf-8') as file:
    models = {name: defs.Model(**data) for name, data in load(file).items()}

model_stats = models[MODEL]
model_path = join(defs.RESULTS_PATH, 'adapter_weights', MODEL)
quantized = model_stats.quantized
prompt_type = model_stats.prompt_type

### Carregamento do dataset

In [ ]:
with open(join(defs.DATA_PATH, 'stt_data', 'test_dataset.json'), 'r', encoding='utf-8') as file:
    test_dataset = [SimpleLesionData(**data) for data in load(file)]

with open(join(defs.DATA_PATH, 'training_dataset_analysis.json'), 'r', encoding='utf-8') as file:
    training_dataset_analysis = SimpleDatasetAnalysis(**load(file))

print(f'Imagens de teste disponíveis: {len(test_dataset)}')

### Navegação — escolha um exemplo

Execute esta célula para listar os exemplos disponíveis por classe.  
Defina `CLASS_FILTER` para filtrar por classe específica (ex.: `'Câncer Melanoma'`) ou `None` para listar todos.  
Anote o índice `[XXXX]` do exemplo que quiser usar e coloque em `EXAMPLE_INDEX` na próxima célula.

In [ ]:
CLASS_FILTER = 'Câncer Melanoma'  # None para listar todos
MAX_SHOW = 50

shown = 0
for i, d in enumerate(test_dataset):
    if CLASS_FILTER is None or d.report.skin_lesion == CLASS_FILTER:
        print(f'[{i:4d}] {d.report.skin_lesion:50s} | risco: {d.report.risk[:10]} | {d.image}')
        shown += 1
        if shown >= MAX_SHOW:
            print(f'... (limitado a {MAX_SHOW}). Aumente MAX_SHOW para ver mais.')
            break

### Carregamento do modelo

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_path,
    load_in_4bit=quantized,
    use_gradient_checkpointing='unsloth',
    device_map={'': 0}
)

FastVisionModel.for_inference(model)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

### Inferência

Altere `EXAMPLE_INDEX` para o índice escolhido na célula de navegação acima e execute.

In [ ]:
EXAMPLE_INDEX = 0  # <--- altere aqui

lesion_data = test_dataset[EXAMPLE_INDEX]
image_path = join(defs.DATA_PATH, 'stt_data', 'images', lesion_data.image)
image = Image.open(image_path).convert('RGB')

print(f'Imagem       : {lesion_data.image}')
print(f'Diagnóstico  : {lesion_data.report.skin_lesion}')
print(f'Risco        : {lesion_data.report.risk}')
print()
ipython_display(image)

formatted_prompt = format_prompt(prompt_type, training_dataset_analysis)
messages = add_inference_message(formatted_prompt)
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

inputs = tokenizer(
    [[image]],
    [input_text],
    add_special_tokens=False,
    return_tensors='pt',
).to('cuda')

output = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=TEMPERATURE,
)

decoded = tokenizer.decode(output[0], skip_special_tokens=True)
response = decoded.split('assistant')[-1].strip()

print()
print('=' * 60)
print('RESPOSTA DO MODELO:')
print('=' * 60)
print(response)